# Module 3: Sparse vs Dense vs Hybrid Search

Qdrant Beginners Course, follow-along notebook.

Course page: https://qdrant.tech/course/beginners/module-3/

## Recap: Modules 1-2

Embeddings + cosine similarity power semantic search. Qdrant stores points (vector + payload) in a collection, uses HNSW for fast search, and payload filters for exact matches.

This module: dense vectors find meaning, sparse (BM25) vectors find exact tokens, hybrid search combines both.


In [ ]:
!pip install -q "qdrant-client[fastembed]" 

## Where dense search struggles

Searching a shoe catalog dense-only for `Nike Pegasus 40`:

| Result | Dense score |
|--------|-------------|
| Nike Pegasus 40 running shoes | 0.8713 |
| Nike Pegasus 41 running shoes | 0.8626 |
| Nike Pegasus 40 womens running shoes | 0.7830 |

The margin between the right shoe and the wrong one is only 0.0087. At scale, that thin margin causes wrong results to outrank the right one.

## Dense vs sparse

Dense: fixed-size vectors (384 dims here), every dimension holds a value. Good for meaning.

Sparse: token-based, only present tokens carry a weight, rest are zero. Good for exact terms. BM25 is the standard sparse model:

```python
# BM25 vector for "Nike Pegasus 40 running shoes"
indices = [1974139272, 24614856, 1784631546, 243905464, 303109060]
values  = [1.67, 1.67, 1.67, 1.67, 1.67]
```

`40` and `41` are different tokens with no relation to sparse search, unlike dense where they're 0.0087 apart. Sparse similarity in Qdrant is always dot product.

Two trained sparse models extend BM25: SPLADE (expands with related terms) and miniCOIL (weights terms by context, recommended for new projects).

## Fusion

Sparse alone flips the ranking (wrong products can tie on shared tokens). Hybrid search runs both retrievers and fuses their ranked lists.

**RRF (Reciprocal Rank Fusion)** merges by rank position, ignoring raw scores (dense and BM25 scores live on different scales). It's the default and safe choice.

**DBSF** normalizes each retriever's score distribution before combining. Use when score magnitude carries information worth keeping.

## Create a hybrid collection

Sparse config needs an `IDF` modifier: it computes the inverse-document-frequency half of BM25 scoring at query time.


In [ ]:
from qdrant_client import QdrantClient, models

client = QdrantClient(
    url="https://YOUR-CLUSTER.cloud.qdrant.io",
    api_key="YOUR_API_KEY",
)

client.create_collection(
    collection_name="products",
    vectors_config={
        "dense": models.VectorParams(size=384, distance=models.Distance.COSINE),
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams(
            modifier=models.Modifier.IDF,
        ),
    },
)

client.create_payload_index(
    collection_name="products",
    field_name="in_stock",
    field_schema=models.PayloadSchemaType.BOOL,
)
client.create_payload_index(
    collection_name="products",
    field_name="sizes",
    field_schema=models.PayloadSchemaType.INTEGER,
)
client.create_payload_index(
    collection_name="products",
    field_name="price",
    field_schema=models.PayloadSchemaType.FLOAT,
)

Create payload indexes before ingesting: Qdrant Cloud strict mode rejects filters on unindexed fields.

## Insert points with both vectors

`models.Document` embeds text locally via FastEmbed before upload.


In [ ]:
CATALOG = [
    (1, "Nike Pegasus 40 running shoes",              139, True,  [8, 9, 10, 11]),
    (2, "Nike Pegasus 41 running shoes",              145, True,  [9, 10, 11]),
    (3, "Nike Pegasus Trail 4 trail running shoes",   149, True,  [9, 10]),
    (4, "Nike Invincible 3 road running shoes",       179, True,  [10, 11]),
    (5, "Adidas Ultraboost 22 running shoes",         189, False, [9, 10]),
    (6, "Brooks Ghost 15 neutral running shoes",      129, True,  [10, 11]),
    (7, "Nike Air Zoom Structure 25 stability shoes", 129, True,  [9, 10]),
    (8, "Nike Pegasus 40 womens running shoes",       139, False, [6, 7, 8]),
]

DENSE_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"
SPARSE_MODEL = "Qdrant/bm25"

client.upsert(
    collection_name="products",
    points=[
        models.PointStruct(
            id=pid,
            vector={
                "dense":  models.Document(text=title, model=DENSE_MODEL),
                "sparse": models.Document(text=title, model=SPARSE_MODEL),
            },
            payload={"title": title, "price": price, "in_stock": stock, "sizes": sizes},
        )
        for pid, title, price, stock, sizes in CATALOG
    ],
)

## Hybrid query with fusion

A `Prefetch` is a sub-query. Fusion merges the candidate lists it returns.


In [ ]:
def hybrid_search(query_text, limit=4):
    return client.query_points(
        collection_name="products",
        prefetch=[
            models.Prefetch(
                query=models.Document(text=query_text, model=DENSE_MODEL),
                using="dense",
                limit=20,   # prefetch limit must be >= outer limit
            ),
            models.Prefetch(
                query=models.Document(text=query_text, model=SPARSE_MODEL),
                using="sparse",
                limit=20,
            ),
        ],
        query=models.RrfQuery(rrf=models.Rrf()),
        limit=limit,
    ).points

for r in hybrid_search("Nike Pegasus 40"):
    print(f"{r.score:.4f}  {r.payload['title']}")

# Real output:
#   1.0000  Nike Pegasus 40 running shoes
#   0.5833  Nike Pegasus 41 running shoes
#   0.5833  Nike Pegasus 40 womens running shoes
#   0.4000  Nike Pegasus Trail 4 trail running shoes

Hybrid widens the margin between the right shoe and its closest rival to 41.7%, versus under 2% for either retriever alone.

## Filtering

Filters work the same way for dense-only, sparse-only, or hybrid. In a hybrid query, put the filter inside each `Prefetch`.


In [ ]:
shopper_filter = models.Filter(
    must=[
        models.FieldCondition(key="in_stock", match=models.MatchValue(value=True)),
        models.FieldCondition(key="sizes",    match=models.MatchValue(value=11)),
    ]
)

def filtered_hybrid_search(query_text, query_filter, limit=4):
    return client.query_points(
        collection_name="products",
        prefetch=[
            models.Prefetch(
                query=models.Document(text=query_text, model=DENSE_MODEL),
                using="dense", filter=query_filter, limit=20,
            ),
            models.Prefetch(
                query=models.Document(text=query_text, model=SPARSE_MODEL),
                using="sparse", filter=query_filter, limit=20,
            ),
        ],
        query=models.RrfQuery(rrf=models.Rrf()),
        limit=limit,
    ).points

for r in filtered_hybrid_search("Nike Pegasus 40", shopper_filter):
    print(f"{r.score:.4f}  {r.payload['title']}")

With no prefetch (single retriever), the filter goes at the top level as `query_filter` instead:


In [ ]:
results = client.query_points(
    collection_name="products",
    query=models.Document(text="Nike Pegasus 40", model=DENSE_MODEL),
    using="dense",
    query_filter=shopper_filter,
    limit=4,
).points

### Try it yourself

1. Query `Nike Pegasus 41`, compare dense-only, sparse-only, and hybrid rankings.
2. Add a `price` range filter (`lte=140`) to `shopper_filter` and rerun.
3. Query `comfortable shoes for long runs`, a phrase no title contains.

## Further reading

- [Hybrid Queries](https://qdrant.tech/documentation/search/hybrid-queries/)
- [Understanding SPLADE and Sparse Vectors](https://qdrant.tech/articles/sparse-vectors/)
- [miniCOIL](https://qdrant.tech/articles/minicoil/)
- [Named Vectors](https://qdrant.tech/documentation/manage-data/vectors/#named-vectors)

Next: `Module4.ipynb`, designing a vector search system that scales.
